# Phase 3: Baseline — Raw Qwen3-4B with Tools

Run the **un-fine-tuned** Qwen3-4B with tool calling enabled to establish "before" metrics.
This gives us failure examples for the talk and a baseline reward score to compare against SFT and RL.

**Expected failures:** wrong tool selection, malformed JSON, infinite looping, no final output, no reasoning.

## Setup

In [ ]:
%pip install -q unsloth trl

In [ ]:
import json
import sys
from pathlib import Path

from unsloth import FastLanguageModel

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT / "nb"))

from bbb.tools import TOOL_FUNCTIONS, TOOL_SCHEMAS
from bbb.helpers__data_gen import (
    SYSTEM_PROMPT,
    TICKERS,
    FOCUS_AREAS,
    make_user_prompt,
    _responses_tool_to_chat,
)
from bbb.helpers__inference import (
    run_local_agent_loop,
    compute_reward,
)

# Derive valid tool names from the tool registry (not hardcoded)
VALID_TOOL_NAMES = set(TOOL_FUNCTIONS.keys())

In [ ]:
# Load Qwen3-4B — no fine-tuning, just the base model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B",
    max_seq_length=8192,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
print(f"Model loaded on {model.device}")

## Tool Setup

Convert tool schemas from Responses API (flat) to Chat Completions (nested) format,
which is what Qwen3's `apply_chat_template(tools=...)` expects.

In [ ]:
from bbb.tools import TOOL_SCHEMAS

# Convert flat Responses API schemas to nested Chat Completions format
TOOLS_CHAT = [_responses_tool_to_chat(t) for t in TOOL_SCHEMAS]

for tool in TOOLS_CHAT:
    fn = tool["function"]
    params = list(fn["parameters"]["properties"].keys())
    print(f"  {fn['name']}({', '.join(params)})")

## Single Ticker Test

Run one ticker with verbose output to see exactly what the raw model does.

In [ ]:
result = run_local_agent_loop(
    model=model,
    tokenizer=tokenizer,
    user_prompt="Research AAPL focusing on financial health and balance sheet strength.",
    system_prompt=SYSTEM_PROMPT,
    tools_chat=TOOLS_CHAT,
    tool_functions=TOOL_FUNCTIONS,
    max_iterations=8,
    enable_thinking=True,
    verbose=True,
)

print(f"\nIterations: {result['n_iterations']}")
print(f"Tool calls: {result['n_tool_calls']}")
print(f"Final output: {len(result['output_text'])} chars")

In [ ]:
# Show the full message flow
for i, msg in enumerate(result["messages"]):
    role = msg["role"].upper()
    content = msg.get("content", "") or ""
    tc = msg.get("tool_calls", [])

    if role == "SYSTEM":
        print(f"[{i}] {role}: {content[:80]}...")
    elif role == "USER":
        print(f"[{i}] {role}: {content}")
    elif role == "ASSISTANT" and tc:
        names = [t["function"]["name"] for t in tc]
        has_think = "<think>" in content
        print(f"[{i}] {role} (think={has_think}, tools={names})")
        if content:
            print(f"    content: {content[:150]}...")
    elif role == "ASSISTANT":
        print(f"[{i}] {role} (final): {content[:200]}...")
    elif role == "TOOL":
        print(f"[{i}] {role}: {len(content)} chars")
    print()

In [ ]:
# Score this trajectory
reward = compute_reward(result["messages"], VALID_TOOL_NAMES)
print("Reward components:")
for k, v in reward.items():
    print(f"  {k:>20}: {v}")

## Batch Evaluation

Run on ~20 tickers to get aggregate baseline metrics.

In [ ]:
import random

# Pick 20 diverse tickers
eval_tickers = random.sample(TICKERS, 20)
print(f"Evaluating: {eval_tickers}")

In [ ]:
baseline_results = []

for ticker in eval_tickers:
    focus = random.choice(FOCUS_AREAS)
    prompt = make_user_prompt(ticker, focus)

    print(f"\n{'='*50}")
    print(f"{ticker} — {focus}")

    try:
        res = run_local_agent_loop(
            model=model,
            tokenizer=tokenizer,
            user_prompt=prompt,
            system_prompt=SYSTEM_PROMPT,
            tools_chat=TOOLS_CHAT,
            tool_functions=TOOL_FUNCTIONS,
            max_iterations=8,
            enable_thinking=True,
            verbose=True,
        )

        reward = compute_reward(res["messages"], VALID_TOOL_NAMES)

        baseline_results.append({
            "ticker": ticker,
            "focus": focus,
            "n_tool_calls": res["n_tool_calls"],
            "n_iterations": res["n_iterations"],
            "output_len": len(res["output_text"]),
            "reward": reward,
            "messages": res["messages"],
        })

        print(f"  → reward={reward['total']}, tool_calls={res['n_tool_calls']}, output={len(res['output_text'])} chars")

    except Exception as e:
        print(f"  → ERROR: {e}")
        baseline_results.append({
            "ticker": ticker,
            "focus": focus,
            "error": str(e),
            "reward": {"total": -3.0},
        })

print(f"\nCompleted: {len(baseline_results)}/{len(eval_tickers)}")

## Results Summary

In [ ]:
# Aggregate metrics
rewards = [r["reward"]["total"] for r in baseline_results]
tool_counts = [r.get("n_tool_calls", 0) for r in baseline_results]
output_lens = [r.get("output_len", 0) for r in baseline_results]
completions = [r["reward"].get("completion", 0) for r in baseline_results]
valid_jsons = [r["reward"].get("valid_json", 0) for r in baseline_results]

print("=== BASELINE RESULTS ===")
print(f"Reward:       avg={sum(rewards)/len(rewards):.2f}  min={min(rewards)}  max={max(rewards)}")
print(f"Tool calls:   avg={sum(tool_counts)/len(tool_counts):.1f}  min={min(tool_counts)}  max={max(tool_counts)}")
print(f"Output chars: avg={sum(output_lens)/len(output_lens):.0f}")
print(f"Completion rate: {sum(completions)/len(completions)*100:.0f}%")
print(f"Valid JSON rate:  {sum(valid_jsons)/len(valid_jsons)*100:.0f}%")

print("\n--- Per-component averages ---")
components = ["valid_json", "thinking", "tool_selection", "efficiency", "completion", "no_hallucination"]
for comp in components:
    vals = [r["reward"].get(comp, 0) for r in baseline_results]
    print(f"  {comp:>20}: {sum(vals)/len(vals):+.2f}")

In [ ]:
# Per-ticker breakdown
print(f"{'Ticker':<8} {'Reward':>7} {'Tools':>6} {'Output':>7} {'JSON':>5} {'Done':>5}")
print("-" * 45)
for r in sorted(baseline_results, key=lambda x: x["reward"]["total"]):
    rw = r["reward"]
    print(
        f"{r['ticker']:<8} {rw['total']:>+7.2f} "
        f"{r.get('n_tool_calls', 0):>6} "
        f"{r.get('output_len', 0):>7} "
        f"{'Y' if rw.get('valid_json') else 'N':>5} "
        f"{'Y' if rw.get('completion') else 'N':>5}"
    )

## Save Baseline Results

Save for comparison with SFT and RL models.

In [ ]:
# Save results (without full messages to keep file small)
output_path = PROJECT_ROOT / "data" / "bbb" / "baseline_results.jsonl"

with open(output_path, "w") as f:
    for r in baseline_results:
        record = {k: v for k, v in r.items() if k != "messages"}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(baseline_results)} results to {output_path}")

# Also save a few full trajectories (failure examples for the talk)
examples_path = PROJECT_ROOT / "data" / "bbb" / "baseline_examples.json"
worst = sorted(baseline_results, key=lambda x: x["reward"]["total"])[:5]
best = sorted(baseline_results, key=lambda x: x["reward"]["total"], reverse=True)[:3]

with open(examples_path, "w") as f:
    json.dump({"worst": worst, "best": best}, f, indent=2, default=str)

print(f"Saved failure/success examples to {examples_path}")